# Fine-Tune FLAN-T5 with RL (PPO) and PEFT to Generate Less-Toxic Summaries

In the 'PEFT_finetuning' notebook, we PEFT fine-tune a FLAN-T5 model for the dialog summarization task. In this notebook, we further finetune it using Reinforcement Learning with Human Feedback using PPO and PEFT. We use Huggingface TRL library's Proximal Policy Optimization(PPO) implementation. We use Meta-AI's hate speech model as the reward model; The reward model is a binary classifier that predicts either "not hate" or "hate" for the given text. Lesser the 'hateful' content, higher the reward. We then train PPO model to fine-tune and reduce the model's toxicity.

## Common installs

In [1]:
# mount gdrive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:

# Install the latest compatible torch and torchdata
#%pip install --no-deps torch==2.5.1 torchdata==0.6.0 --quiet

%pip install --no-deps torch torchdata --quiet

# Install other updated packages (latest stable versions)
%pip install -U \
    datasets \
    transformers \
    peft --quiet

%pip install evaluate --quiet
%pip install rouge_score --quiet

# TRL installation
%pip install --no-deps trl==0.11
%pip install tyro --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, GenerationConfig
from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType

# trl: Transformer Reinforcement Learning library
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model
from trl.core import LengthSampler

import torch
import evaluate

import numpy as np
import pandas as pd
import os.path

from tqdm import tqdm
tqdm.pandas()

CONST = {
  'DATASET_NAME' : 'knkarthick/dialogsum',
  'MODEL_NAME' : 'google/flan-t5-base',
  'INSTRUCT_MODEL_PATH' : 'truocpham/flan-dialogue-summary-checkpoint',
  'PEFT_CHECKPOINT_PATH' : '/content/drive/MyDrive/Colab Notebooks/Gen AI Notebooks/summary-model/checkpoints/peft-dialogue-summary-checkpoint/',
  'SUMMARY_RESULT_FILE': '/content/drive/MyDrive/Colab Notebooks/Gen AI Notebooks/summary-model/data/dialogue-summary-training-results.csv'
}

#set random seed

def set_seeds(seed=123):
  os.environ['PYTHONHASHSEED']=str(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  np.random.seed(seed)

set_seeds()

rouge = evaluate.load('rouge')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load the dataset and tokenize

In [4]:
model_name = CONST['MODEL_NAME']
dataset_original = load_dataset(CONST['DATASET_NAME'])
dataset_original

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
dataset_original['train']['dialogue'][0]

"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor."

In [6]:
#helper function to generate summary prompt given the dialog
def get_summary_prompt(dialogue):
  prompt = f"""
  Summarize the following conversation.

  {dialogue}

  Summary:
  """
  return prompt

In [7]:
'''
Loads the training dataset, filters it by min and max allowable summary length, an tokenizes it using the given model_name.
Returns the tokenized dataset.
'''
def build_dataset(model_name,
                  dataset_name,
                  input_min_text_length,
                  input_max_text_length):


    # load train dataset
    dataset = load_dataset(dataset_name, split="train")

    # Filter the dialogues of length between input_min_text_length and input_max_text_length characters.
    dataset = dataset.filter(lambda x: len(x["dialogue"]) > input_min_text_length and len(x["dialogue"]) <= input_max_text_length, batched=False)

    # Prepare tokenizer. Setting device_map="auto" allows to switch between GPU and CPU automatically.
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")

    def tokenize(sample):

        # Wrap each dialogue with the instruction.
        prompt = get_summary_prompt(sample["dialogue"])

        sample["input_ids"] = tokenizer.encode(prompt)

        # This must be called "query", which is a requirement of our PPO library.
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    # Tokenize each dialogue.
    dataset = dataset.map(tokenize, batched=False)
    dataset.set_format(type="torch")

    # Split the dataset into train and test parts.
    dataset_splits = dataset.train_test_split(test_size=0.2, shuffle=False, seed=42)

    return dataset_splits

dataset = build_dataset(model_name=model_name,
                        dataset_name=CONST['DATASET_NAME'],
                        input_min_text_length=200,
                        input_max_text_length=1000)

print(dataset)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10022 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 8017
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 2005
    })
})


We will use the PEFT-finetuning-checkpoint created in the previous notebook to load the PEFT finetuned model. We will then apply RL(PPO) on this model using PEFT technique.

Add the adapter to the original FLAN-T5 model. In the previous lab you were adding the fully trained adapter only for inferences, so there was no need to pass LoRA configurations doing that. Now you need to pass them to the constructed PEFT model, also putting `is_trainable=True`.

## Load PEFT fine-tuned model and build PPO model

In [8]:
#helper function to print number of trainable parameters
def show_trainable_parameters_details(model):
    trainable_params_cnt = 0
    total_params_cnt = 0

    for _, param in model.named_parameters():
        total_params_cnt += param.numel()
        if param.requires_grad:
            trainable_params_cnt += param.numel()

    return f"trainable parameters: {trainable_params_cnt}\n total parameters: {total_params_cnt}\n % of trainable model parameters: {100 * trainable_params_cnt / total_params_cnt:.2f}%"


In [9]:
#load PEFT fine-tuned model from the local checkpoint (Created in the previous notebook)
device = 0 if torch.cuda.is_available() else "cpu"

lora_config = LoraConfig(
    r=32, # Rank
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name,
                                              torch_dtype=torch.bfloat16)

peft_model = PeftModel.from_pretrained(model,
                                       CONST['PEFT_CHECKPOINT_PATH'],
                                       lora_config=lora_config,
                                       torch_dtype=torch.bfloat16,
                                       device_map="auto",
                                       is_trainable=True)

print(f'PEFT model parameters to be updated:\n{show_trainable_parameters_details(peft_model)}\n')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PEFT model parameters to be updated:
trainable parameters: 3538944
 total parameters: 251116800
 % of trainable model parameters: 1.41%



In [10]:
#create PPO model from the PEFT model. Use bfloat15 datatype.
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(peft_model,
                                                               torch_dtype=torch.bfloat16,
                                                               is_trainable=True)

print(f'PPO model parameters to be updated (ValueHead + 769 params):\n{show_trainable_parameters_details(ppo_model)}\n')
print(ppo_model.v_head)

PPO model parameters to be updated (ValueHead + 769 params):
trainable parameters: 3539713
 total parameters: 251117569
 % of trainable model parameters: 1.41%

ValueHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (summary): Linear(in_features=768, out_features=1, bias=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
)


Note: The PPO model architecture is same as PEFT model except the additional value head, which is simply a linear combinatio of the final layer's output (with 768 neurons + a bias term). Thus,

```
# of trainable params of PPO_model = # of trainable params of PEFT_model + # of params of value_head,

where, # of params of value_head = 768*1 + 1 = 769
```


Note that the PPO algorithm requires a frozen reference model (to compare KL divergence ). Thus we create a frozen copy of the PPO_model which will not be fine-tuned - a reference model. The reference model will represent the LLM before detoxification and it is non-trainable.

In [11]:
ref_model = create_reference_model(ppo_model)

print(f'Reference model parameters to be updated:\n{show_trainable_parameters_details(ref_model)}\n')

Reference model parameters to be updated:
trainable parameters: 0
 total parameters: 251117569
 % of trainable model parameters: 0.00%



## Prepare Reward Model

We use [Meta AI's RoBERTa-based hate speech model](https://huggingface.co/facebook/roberta-hate-speech-dynabench-r4-target) as the reward model. It accepts text(s) and outputs logits/probability of the two output labels: 'hate' and 'nonhate'. We use these scores in two different ways.

1. As reward in PPO: Since we use reward based model, we use logits of 'nonhate' score as the reward in the PPO process. Thus, higher logit (positive score) implies higher probability of 'nonhateful' or non-toxic content, and thus higher reward. On the other hand, lower logit (negative value) implies higher probability of 'hateful' or 'toxic' content and hence lower reward.

2. Toxicity evaulation: We use probability of 'hate' lable as the indication of the toxicity of the text. This can be used to evaulate overall toxicity of the given text.

For convenience, we create a helper class encapsulating a couple of useful APIs.

```
#To collect reward for the given text (-ve for hateful, +ve for non-hateful):
toxicity_reward_helper.getscore(text)

#To get toxicity score for evaulation (0 to 1):
toxicity_reward_helper.evaltoxicity(text)
```

In [32]:
class ToxicityRewardHelper():

  def __init__(self):
    self.model_name = "facebook/roberta-hate-speech-dynabench-r4-target"

    self.sentiment_pipe = pipeline("sentiment-analysis",
                              model=self.model_name,
                              device=device,
                              framework="pt")
    self.logits_kwargs = {
        "top_k": None, # Return all scores.
        "function_to_apply": "none", # Set to "none" to retrieve raw logits.
        "batch_size": 16
    }

    self.prob_kwargs = {
        "top_k": None, # Return all scores.
        "function_to_apply": "softmax", # Set to "softmax" to apply softmax and retrieve probabilities.
        "batch_size": 16
    }

    self.toxicity_evaluator = evaluate.load("toxicity",
                                    self.model_name,
                                    module_type="measurement",
                                    toxic_label="hate")

  def getscore(self, text, label='nothate', retprob=False):
    if retprob:
      ret = self.sentiment_pipe(text, **self.prob_kwargs)
    else:
      ret = self.sentiment_pipe(text, **self.logits_kwargs)

    #single text string
    if type(text) == str:
      ret = [torch.tensor(item["score"]) for item in ret if item['label'] == label]
    else:
      ret = [torch.tensor(item["score"]) for arritem in ret for item in arritem if item['label'] == label]


    return ret

  def evaltoxicity(self, text):
    return self.toxicity_evaluator.compute(predictions=[text])["toxicity"]

toxicity_reward_helper = ToxicityRewardHelper()

Device set to use cuda:0
Device set to use cuda:0


In [13]:
# Test a couple of sentences and check the output of the reward helper.
# Note that we would expect the second sentences to be given higher probability of hateful content, and thus higher toxicit score.
sentences = ["#Person 1# tells Tommy that he didn't like the movie.", "#Person 1# tells Tommy that the movie was terrible, dumb and stupid."]

for sentence in sentences:
  print(f'sentence: {sentence}')
  print(f"toxic_logit: {toxicity_reward_helper.getscore(sentence,label='hate')}, nottoxic_logit: {toxicity_reward_helper.getscore(sentence,label='nothate')}")
  print(f"toxic_prob: {toxicity_reward_helper.getscore(sentence,label='hate',retprob=True)}, nottoxic_prob: {toxicity_reward_helper.getscore(sentence,label='nothate',retprob=True)}")
  print(f'toxicity score: {toxicity_reward_helper.evaltoxicity(sentence)}')
  print('\n')


sentence: #Person 1# tells Tommy that he didn't like the movie.
toxic_logit: [tensor(-2.4896)], nottoxic_logit: [tensor(3.1141)]
toxic_prob: [tensor(0.0037)], nottoxic_prob: [tensor(0.9963)]
toxicity score: {'toxicity': [0.0036706076934933662]}


sentence: #Person 1# tells Tommy that the movie was terrible, dumb and stupid.
toxic_logit: [tensor(0.3723)], nottoxic_logit: [tensor(-0.6921)]
toxic_prob: [tensor(0.7435)], nottoxic_prob: [tensor(0.2565)]
toxicity score: {'toxicity': [0.7435277104377747]}




In [1]:
'''
This function generates summaries for the given dataset using the given model,
and evaualtes the generated summaries using the given toxicity_reward_model.

It then returns the mean and standard deviation of the toxicity scores for the generated summaries.

Assuming that GPU is avaiable, this function explicitly moves model and tensors on device=0. This is because only PPO model
is loaded on GPU (assuming GPU is available) for training, but the original PEFT model is still loaded on CPU.
'''

def evaluate_dataset_toxicity(
    model,
    toxicity_reward_helper,
    tokenizer,
    dataset,
    num_samples
):
    max_new_tokens = 100
    toxicities = []
    input_texts = []

    # Move model to device
    model.to(device)
    model.eval()

    for i, sample in tqdm(enumerate(dataset), total=num_samples):
        if i >= num_samples:
            break

        input_text = sample["query"]
        input_texts.append(input_text)

        # Tokenize input
        encoded = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        input_ids = encoded["input_ids"].to(device)
        #attention_mask = encoded["attention_mask"].to(device)

        # Generate response
        generation_config = GenerationConfig(
            max_new_tokens=max_new_tokens,
            top_k=0,
            top_p=1.0,
            do_sample=True
        )

        with torch.no_grad():
            response_token_ids = model.generate(
                input_ids=input_ids,
                #attention_mask=attention_mask,
                generation_config=generation_config
            )

        generated_text = tokenizer.decode(
            response_token_ids[0],
            skip_special_tokens=True
        )

        full_text = input_text + " " + generated_text

        toxicities.append(toxicity_reward_helper.evaltoxicity(full_text))

    # Compute statistics
    toxicity_mean = np.mean(toxicities)
    toxicity_std = np.std(toxicities)

    return toxicity_mean, toxicity_std


And now perform the calculation of the model toxicity before fine-tuning/detoxification:

In [34]:
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")

mean_before_detoxification, std_before_detoxification = evaluate_dataset_toxicity(model=ref_model,
                                                                          toxicity_reward_helper=toxicity_reward_helper,
                                                                          tokenizer=tokenizer,
                                                                          dataset=dataset["test"],
                                                                          num_samples=10)

print(f'toxicity [mean, std] before detox: [{mean_before_detoxification}, {std_before_detoxification}]')

100%|██████████| 10/10 [00:18<00:00,  1.80s/it]

toxicity [mean, std] before detox: [0.03795389843871817, 0.0392655161587778]


In [35]:
# %pip show numpy | grep Version:
# %pip show transformers | grep Version:
# %pip show trl | grep Version:

## PPO finetuning


In [36]:
#collator class
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

test_data = [{"key1": "value1", "key2": "value2", "key3": "value3"}]
print(f'Collator input: {test_data}')
print(f'Collator output: {collator(test_data)}')

Collator input: [{'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}]
Collator output: {'key1': ['value1'], 'key2': ['value2'], 'key3': ['value3']}


Set up the configuration parameters. The PPO_model is optimized while the ref_model serves as a reference to calculate the KL-divergence from the starting point. This works as an additional reward signal in the PPO training to make sure the optimized model does not deviate too much from the original LLM.

In [37]:
learning_rate=1.41e-5
max_ppo_epochs=1
mini_batch_size=4
batch_size=16

#setup config ad trainer obects
config = PPOConfig(
    model_name=model_name,
    learning_rate=learning_rate,
    ppo_epochs=max_ppo_epochs,
    mini_batch_size=mini_batch_size,
    batch_size=batch_size
)

ppo_trainer = PPOTrainer(config=config,
                         model=ppo_model,
                         ref_model=ref_model,
                         tokenizer=tokenizer,
                         dataset=dataset["train"],
                         data_collator=collator)

### Fine-Tune the Model

The fine-tuning loop consists of the following main steps:
1. Generate query responses from the trainer
2. Get reard for query+responses from the reward model.
3. Optimize policy with PPO using the (query, response, reward) triplet.

In [38]:
output_min_length = 100
output_max_length = 400
output_length_sampler = LengthSampler(output_min_length, output_max_length)

generation_kwargs = {
    "min_length": 10,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True
}

max_ppo_steps = 10

for step, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    # Break when you reach max_steps.
    if step >= max_ppo_steps:
        break

    prompt_tensors = batch["input_ids"]

    # Get response from FLAN-T5/PEFT LLM.
    summary_tensors = []

    for prompt_tensor in prompt_tensors:
        max_new_tokens = output_length_sampler()

        generation_kwargs["max_new_tokens"] = max_new_tokens
        summary = ppo_trainer.generate(prompt_tensor, **generation_kwargs)

        summary_tensors.append(summary.squeeze()[-max_new_tokens:])

    # This needs to be called "response".
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in summary_tensors]

    # Compute reward outputs.
    query_response_pairs = [q + r for q, r in zip(batch["query"], batch["response"])]
    reward_tensors = toxicity_reward_helper.getscore(query_response_pairs)

    # Run PPO step.
    stats = ppo_trainer.step(prompt_tensors, summary_tensors, reward_tensors)
    ppo_trainer.log_stats(stats, batch, reward_tensors)

    print(f'objective/kl: {stats["objective/kl"]}')
    print(f'ppo/returns/mean: {stats["ppo/returns/mean"]}')
    print(f'ppo/policy/advantages_mean: {stats["ppo/policy/advantages_mean"]}')
    print('-'.join('' for x in range(100)))

1it [00:33, 33.64s/it]

objective/kl: 29.83519172668457
ppo/returns/mean: -0.7544234395027161
ppo/policy/advantages_mean: 0.02710779756307602
---------------------------------------------------------------------------------------------------


2it [01:21, 42.18s/it]

objective/kl: 25.060611724853516
ppo/returns/mean: -0.4350373148918152
ppo/policy/advantages_mean: 0.013059448450803757
---------------------------------------------------------------------------------------------------


3it [02:09, 44.58s/it]

...

objective/kl: 18.989452362060547
ppo/returns/mean: -0.24842330813407898
ppo/policy/advantages_mean: -0.0016221404075622559
---------------------------------------------------------------------------------------------------


9it [06:44, 42.17s/it]

objective/kl: 20.720653533935547
ppo/returns/mean: -0.3257753252983093
ppo/policy/advantages_mean: -0.0019097027834504843
---------------------------------------------------------------------------------------------------


10it [07:27, 44.76s/it]

objective/kl: 19.981369018554688
ppo/returns/mean: -0.18215681612491608
ppo/policy/advantages_mean: -0.003693327307701111
---------------------------------------------------------------------------------------------------


### Evaluate the Model Quantitatively


In [39]:
mean_after_detoxification, std_after_detoxification = evaluate_dataset_toxicity(model=ppo_model,
                                                                        toxicity_reward_helper = toxicity_reward_helper,
                                                                        tokenizer=tokenizer,
                                                                        dataset=dataset["test"],
                                                                        num_samples=10)
print(f'toxicity [mean, std] after detox: [{mean_after_detoxification}, {std_after_detoxification}]')

100%|██████████| 10/10 [00:13<00:00,  1.36s/it]

toxicity [mean, std] after detox: [0.033459144085645674, 0.055162481034511684]


And compare the toxicity scores of the reference model (before detoxification) and fine-tuned model (after detoxification).

In [40]:
mean_improvement = (mean_before_detoxification - mean_after_detoxification) / mean_before_detoxification
std_improvement = (std_before_detoxification - std_after_detoxification) / std_before_detoxification

print(f'Percentage improvement of toxicity score after detoxification:')
print(f'mean: {mean_improvement*100:.2f}%')
print(f'std: {std_improvement*100:.2f}%')

Percentage improvement of toxicity score after detoxification:
mean: 11.84%
std: -40.49%


### Evaluate the Model Qualitatively


In [43]:
batch_size = 20
compare_results = {}

df_batch = dataset["test"][0:batch_size]

compare_results["query"] = df_batch["query"]
prompt_tensors = df_batch["input_ids"]

summary_tensors_ref = []
summary_tensors = []

# Get response from ppo and base model.
for i in tqdm(range(batch_size)):
    gen_len = output_length_sampler()
    generation_kwargs["max_new_tokens"] = gen_len

    summary = ref_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwargs
    ).squeeze()[-gen_len:]
    summary_tensors_ref.append(summary)

    summary = ppo_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwargs
    ).squeeze()[-gen_len:]
    summary_tensors.append(summary)

# Decode responses.
compare_results["response_before"] = [tokenizer.decode(summary_tensors_ref[i]) for i in range(batch_size)]
compare_results["response_after"] = [tokenizer.decode(summary_tensors[i]) for i in range(batch_size)]

# Sentiment analysis of query/response pairs before/after.
texts_before = [d + s for d, s in zip(compare_results["query"], compare_results["response_before"])]
compare_results["reward_before"] = [score_tensor.item() for score_tensor in toxicity_reward_helper.getscore(texts_before)]

texts_after = [d + s for d, s in zip(compare_results["query"], compare_results["response_after"])]
compare_results["reward_after"] = [score_tensor.item() for score_tensor in toxicity_reward_helper.getscore(texts_after)]

100%|██████████| 20/20 [01:19<00:00,  3.97s/it]


Store and review the results in a DataFrame

In [44]:
pd.set_option('display.max_colwidth', 500)
df_compare_results = pd.DataFrame(compare_results)
df_compare_results["reward_diff"] = df_compare_results['reward_after'] - df_compare_results['reward_before']
df_compare_results_sorted = df_compare_results.sort_values(by=['reward_diff'], ascending=False).reset_index(drop=True)
df_compare_results_sorted

,query,response_before,response_after,reward_before,reward_after,reward_diff
0,"Summarize the following conversation. #Person1#: Oh, my God! What's this? #Person2#: What? #Person1#: Look! This window is open. #Person2#: Did you open it before we left? #Person1#: Are you kidding? It's winter. Why would I open it? #Person2#: I don't know. Wait. Is this yours? #Person1#: No! Oh, my God! Someone has broken into the house. #Person2#: It looks that way. That's probably why the door wasn't locked when we came in. #Person1#: I locked it when I left though. #Person2#: Yes, but t...","<pad> Allen doubts about the way the window worked out and invites #Person1# to look upstairs. He tells #Person1# that the robber broke in and he left through the door, so nobody will ever see him.</s>","<pad> Allen and #Person2# decide to open the door because someone broke inside and stole the remote controller. Allen wants to look upstairs, but he cannot find someone there.</s>",1.893058,2.608392,0.715334
1,"Summarize the following conversation. #Person1#: Could you help me, Sir? My flight got in 15 minutes ago. Everyone else has picked up the luggage but mine hasn't come through. #Person2#: I'm sorry, Madam, I'll go and find out if there is any more to come. Summary: </s>",<pad> #Person2# will be going to check on the luggage but #Person1# was not picked up.</s>,"<pad> #Person1# had a flight with people, but it hasn't come in.</s>",2.283696,2.784103,0.500407
2,"Summarize the following conversation. #Person1#: So how did you like the restaurant? #Person2#: Actually, it could have been better. #Person1#: What didn't you like about it? #Person2#: It is a new restaurant. I don't think they have their act together yet. #Person1#: What did you think about the food? #Person2#: I felt that the food was pretty mediocre. #Person1#: The service wasn't that great, either. #Person2#: I agree. The service was not good. #Person1#: Do you think that you want to tr...",<pad> #Person1# and #Person2# change their need for any kind of satisfaction from a new restaurant after the meal airopned it more bad. But they think the food is lacking in quality and service was bad.</s>,<pad> #Person2# could have been worse but the staff was not as good. She finds she's had too much of the food and service.</s>,1.749464,2.220500,0.471036
3,"Summarize the following conversation. #Person1#: I'd like to have this cashed, please. #Person2#: Please put you name and address here. May I see your passport? #Person1#: Yes. #Person2#: How would you like it? #Person1#: Ten hundreds and ten twenties, and the rest in small change, please. #Person2#: OK. Here you are. Summary: </s>","<pad> #Person1# asks #Person2# for a cashed bus. #Person1# gives #Person2# name, address, and passport.</s>","<pad> #Person1# wants to have the money cashed and buy some small change, $10 and $100.</s>",1.744690,2.128138,0.383447
4,"Summarize the following conversation. #Person1#: How much are you asking for this? #Person2#: I'm offering them to you at 150 yuan a piece. Is that all right? #Person1#: Is tax already included in their price? #Person2#: Yes. Our price can't be matched. #Person1#: Would you consider a volume discount? #Person2#: If you buy 1, 000 or more, you'll get a 10 % discount. #Person1#: I'll accept your offer. Summary: </s>","<pad> #Person1# asks for an onion with a price of 150 yuan and #Person2# offers it to #Person1# at a 10 % discount for amply buying, providing coverage and opening discounts.</s>","<pad> Qing&#Person2# sells the Pebao plate with a price of 150 yuan a piece, and B doesn't accept the offer.</s>",2.431055,2.729479,0.298424
5,"Summarize the following conversation. #Person1#: Let's take a coffee break, shall we? #Person2#: I wish I could, but I can't. #Person1#: What keeps you so busy? You've been sitting there for hours. You've got to walk around. You just can't stay on the computer forever. #Person2#: Well, I am up to my neck in work. I've got to finish this report. Sarah needs it

## Conclusion:

We observe that after RLHF fine-tuning, the mean toxicity score in a small sample of 20 summaries reduced by aroud ~12%. We do notice some increase in the standard deviation of the scores, but that does not seem like a concern.